# Logistic Regression and Evaluation — Solution

**Dataset:** Swiss avalanche accident records; fatal = 1 if at least one person died.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay

## 1. Calculate a probability

For score 1, replace the gap with `1 / (1 + np.exp(-score))`. Which label follows at threshold 0.5?

In [ ]:
score = 1
p = 1 / (1 + np.exp(-score))
print('Probability:', round(p, 3))
print('Class at threshold 0.5:', int(p >= 0.5))

**Answer:** The probability is approximately 0.731, so the predicted class is 1.

## 2. Load and split accident records

Preparation is provided. How is a positive case defined? At what time is the input available?

In [ ]:
aval = pd.read_csv('../00_Data/avalanche_accidents_switzerland.csv')
aval = aval[aval['hydrological_year'] >= '2000/01'].copy()
aval = aval.dropna(subset=['fully_buried', 'fatal'])
X = aval[['fully_buried']]
y = aval['fatal']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Accidents:', len(aval), 'Share fatal:', round(y.mean(), 3))
print(X_train.head())

**Answer:** Positive means at least one death. The number fully buried is post-accident information. This is a teaching model for outcomes of recorded accidents, not pre-trip avalanche prediction.

## 3. Train and see the sigmoid

Replace the gap with `LogisticRegression()`. Does more burial correspond to higher predicted probability?

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)
print('Intercept:', model.intercept_, 'Slope:', model.coef_)
grid = pd.DataFrame({'fully_buried': np.linspace(0, 8, 100)})
plt.plot(grid['fully_buried'], model.predict_proba(grid)[:, 1])
plt.xlabel('Number of fully buried persons')
plt.ylabel('P(fatal accident)')
plt.show()
probability = model.predict_proba(X_test)[:, 1]

**Answer:** A positive slope gives an increasing curve: the model estimates higher probability with more fully buried people. This is an association within the recorded accidents.

## 4. Compute precision and recall by hand

Use slide 48 counts. Replace the gap with `TP / (TP + FN)`. Explain precision versus recall.

In [ ]:
TN, FP, FN, TP = 75, 22, 24, 50
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print('Precision:', round(precision, 3))
print('Recall:', round(recall, 3))
print('FPR:', round(FP / (FP + TN), 3))

**Answer:** Precision ≈ 0.694 is the share of predicted positives that are correct; recall ≈ 0.676 is the share of actual positives found.

## 5. Evaluate the model

The evaluation is provided. Read the matrix rows and columns, and compare accuracy with the always-negative baseline. Why is recall also needed?

In [ ]:
predicted = (probability >= 0.5).astype(int)
cm = confusion_matrix(y_test, predicted, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['Non-fatal', 'Fatal']).plot()
plt.show()
print('Accuracy:', round(accuracy_score(y_test, predicted), 3))
print('Precision:', round(precision_score(y_test, predicted, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, predicted), 3))
print('F1:', round(f1_score(y_test, predicted), 3))
print('Always-negative accuracy:', round(1 - y_test.mean(), 3))

**Answer:** Rows show actual outcomes, columns predicted outcomes. The always-negative model misses every fatal case despite potentially high accuracy. Recall tells us how many fatal cases this model identifies.

## 6. Compare two fixed thresholds

This comparison is supplied. What changes when the threshold goes from 0.5 to 0.3? Costs of 10 per missed positive and 1 per false alarm are illustrative. Do not choose a new threshold from this test comparison.

In [ ]:
for threshold in [0.5, 0.3]:
    labels = (probability >= threshold).astype(int)
    TN, FP, FN, TP = confusion_matrix(y_test, labels, labels=[0, 1]).ravel()
    print('Threshold:', threshold, 'FP:', FP, 'FN:', FN,
          'recall:', round(TP / (TP + FN), 3),
          'illustrative cost:', 10 * FN + FP)

**Answer:** A lower threshold cannot increase missed positives, but it may increase false alarms. The preferred threshold depends on costs and should be decided using validation data.

## 7. Read ROC and AUC

The plot is provided. Is the curve above the random-ranking diagonal? Explain AUC without referring to a single threshold.

In [ ]:
RocCurveDisplay.from_predictions(y_test, probability)
plt.plot([0, 1], [0, 1], 'k--', label='Random ranking')
plt.legend()
plt.show()
print('Test AUC:', round(roc_auc_score(y_test, probability), 3))

**Answer:** A curve above the diagonal indicates useful ranking on these rows. AUC summarises how well positives rank above negatives across thresholds; it is not the percentage of correct class labels.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))